In [2]:
#Phase1: Data collection
# Cleaning the data from 2017-09-01 to 2025-09-23
import pandas as pd 
import numpy as np  
df = pd.read_csv('/home/dipankar/Desktop/crypto_price_prediction/data/raw/BTCUSDT_5m_2017-09-01_to_2025-09-23.csv')
print(df.head())
df.shape
df.info()
df.describe()

           datetime     open     high      low    close    volume
0  2017-09-01 05:30  4689.89  4689.91  4689.89  4689.91  0.422439
1  2017-09-01 05:35  4725.00  4725.00  4725.00  4725.00  0.473560
2  2017-09-01 05:40  4731.12  4731.13  4701.10  4731.13  1.291685
3  2017-09-01 05:45  4703.02  4705.02  4703.02  4705.02  0.097104
4  2017-09-01 05:50  4710.02  4710.02  4706.06  4706.06  0.541234
<class 'pandas.DataFrame'>
RangeIndex: 846445 entries, 0 to 846444
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   datetime  846445 non-null  str    
 1   open      846445 non-null  float64
 2   high      846445 non-null  float64
 3   low       846445 non-null  float64
 4   close     846445 non-null  float64
 5   volume    846445 non-null  float64
dtypes: float64(5), str(1)
memory usage: 38.7 MB


,open,high,low,close,volume
count,846445.000000,846445.000000,846445.000000,846445.000000,846445.000000
mean,33929.335508,33967.740793,33890.129707,33929.448574,223.392037
std,30239.636750,30263.813164,30215.054199,30239.748578,386.670105
min,2830.000000,2878.000000,2817.000000,2820.000000,0.000000
25%,9092.230000,9102.850000,9081.370000,9092.130000,55.199344
50%,24132.950000,24170.000000,24094.000000,24132.900000,108.675830
75%,49773.580000,49854.080000,49677.270000,49772.580000,226.914370
max,124243.310000,124474.000000,124124.990000,124243.320000,15659.934840


In [3]:
#phase2: Data cleaning 
#Find the missing values and the duplicatesin the dataset
print("Number of The missing values in the dataset:")
print(df.isnull().sum())

print("Number of duplicates in the dataset:")
print(df.duplicated().sum())


Number of The missing values in the dataset:
datetime    0
open        0
high        0
low         0
close       0
volume      0
dtype: int64
Number of duplicates in the dataset:
0


In [4]:
#Phase 3: If there are any missing values or duplicates, we will drop them from the dataset
#Dropping the missing values and duplicates from the dataset
df = df.dropna()
print("Number of The missing values in the dataset after dropping:")
print(df.isnull().sum())
#Dropping the duplicates from the dataset
df = df.drop_duplicates()
print("Number of duplicates in the dataset after dropping:")
print(df.duplicated().sum())
print("First 5 rows of the cleaned dataset:")
print(df.head())

Number of The missing values in the dataset after dropping:
datetime    0
open        0
high        0
low         0
close       0
volume      0
dtype: int64
Number of duplicates in the dataset after dropping:
0
First 5 rows of the cleaned dataset:
           datetime     open     high      low    close    volume
0  2017-09-01 05:30  4689.89  4689.91  4689.89  4689.91  0.422439
1  2017-09-01 05:35  4725.00  4725.00  4725.00  4725.00  0.473560
2  2017-09-01 05:40  4731.12  4731.13  4701.10  4731.13  1.291685
3  2017-09-01 05:45  4703.02  4705.02  4703.02  4705.02  0.097104
4  2017-09-01 05:50  4710.02  4710.02  4706.06  4706.06  0.541234


In [5]:
#phase 4: Adding new features to the dataset
#Deleting the returns of the close price from the dataset
df = df.drop(columns=[
    "return_5m",
    "return_10m",
    "return_15m",
    "return_30m",
    "return_1h",
    "log_return_10m",
    "log_return_30m",
    "log_return_1h",
    "target_return_5m",
    "target_return_15m",
],errors='ignore')
#Adding the log returns of the close price to the dataset
df["log_return_5m"] = np.log(df["close"] / df["close"].shift(1))

df["log_return_15m"] = np.log(df["close"] / df["close"].shift(3))

print("First 5 rows of the dataset with new features:")
print(df.head())

First 5 rows of the dataset with new features:
           datetime     open     high      low    close    volume  \
0  2017-09-01 05:30  4689.89  4689.91  4689.89  4689.91  0.422439   
1  2017-09-01 05:35  4725.00  4725.00  4725.00  4725.00  0.473560   
2  2017-09-01 05:40  4731.12  4731.13  4701.10  4731.13  1.291685   
3  2017-09-01 05:45  4703.02  4705.02  4703.02  4705.02  0.097104   
4  2017-09-01 05:50  4710.02  4710.02  4706.06  4706.06  0.541234   

   log_return_5m  log_return_15m  
0            NaN             NaN  
1       0.007454             NaN  
2       0.001297             NaN  
3      -0.005534        0.003217  
4       0.000221       -0.004017  


In [6]:
#Phase 5: Adding the Technical indicators to the dataset
#Adding the momentum, trend and volatility 
from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands
# RSI
rsi = RSIIndicator(df['close'], window=14)
df['rsi'] = rsi.rsi()

# MACD
macd = MACD(df['close'], window_slow=26, window_fast=12, window_sign=9, fillna=False)
df['macd'] = macd.macd()
# Bollinger Bands
bollinger = BollingerBands(df['close'], window=20, window_dev=2)
df['bollinger_hband'] = bollinger.bollinger_hband()
df['bollinger_lband'] = bollinger.bollinger_lband()
df['bollinger_mavg'] = bollinger.bollinger_mavg()
print("First 5 rows of the dataset with new features:")
print(df.head(45))

First 5 rows of the dataset with new features:
            datetime     open     high      low    close    volume  \
0   2017-09-01 05:30  4689.89  4689.91  4689.89  4689.91  0.422439   
1   2017-09-01 05:35  4725.00  4725.00  4725.00  4725.00  0.473560   
2   2017-09-01 05:40  4731.12  4731.13  4701.10  4731.13  1.291685   
3   2017-09-01 05:45  4703.02  4705.02  4703.02  4705.02  0.097104   
4   2017-09-01 05:50  4710.02  4710.02  4706.06  4706.06  0.541234   
5   2017-09-01 05:55  4710.01  4731.13  4710.01  4711.26  1.480369   
6   2017-09-01 06:00  4707.27  4745.35  4707.27  4715.00  2.858978   
7   2017-09-01 06:05  4706.09  4715.00  4706.09  4715.00  0.371760   
8   2017-09-01 06:10  4715.00  4715.00  4715.00  4715.00  0.000000   
9   2017-09-01 06:15  4718.11  4737.76  4718.07  4718.07  1.486984   
10  2017-09-01 06:20  4718.07  4725.82  4717.68  4717.68  3.417535   
11  2017-09-01 06:25  4718.88  4725.52  4718.88  4721.05  3.270025   
12  2017-09-01 06:30  4730.05  4730.05  471

In [7]:
#Phase 6: Adding the target variable to the dataset
df["target_log_return_5m"] = np.log(
    df["close"].shift(-1) / df["close"]
)
df["target_log_return_15m"] = np.log(
    df["close"].shift(-3) / df["close"]
)
print("First 45 rows of the dataset with new features:")
print(df.head(45))

First 45 rows of the dataset with new features:
            datetime     open     high      low    close    volume  \
0   2017-09-01 05:30  4689.89  4689.91  4689.89  4689.91  0.422439   
1   2017-09-01 05:35  4725.00  4725.00  4725.00  4725.00  0.473560   
2   2017-09-01 05:40  4731.12  4731.13  4701.10  4731.13  1.291685   
3   2017-09-01 05:45  4703.02  4705.02  4703.02  4705.02  0.097104   
4   2017-09-01 05:50  4710.02  4710.02  4706.06  4706.06  0.541234   
5   2017-09-01 05:55  4710.01  4731.13  4710.01  4711.26  1.480369   
6   2017-09-01 06:00  4707.27  4745.35  4707.27  4715.00  2.858978   
7   2017-09-01 06:05  4706.09  4715.00  4706.09  4715.00  0.371760   
8   2017-09-01 06:10  4715.00  4715.00  4715.00  4715.00  0.000000   
9   2017-09-01 06:15  4718.11  4737.76  4718.07  4718.07  1.486984   
10  2017-09-01 06:20  4718.07  4725.82  4717.68  4717.68  3.417535   
11  2017-09-01 06:25  4718.88  4725.52  4718.88  4721.05  3.270025   
12  2017-09-01 06:30  4730.05  4730.05  47

In [8]:
#Phase 7: Dropping the missing values from the dataset
df = df.dropna().reset_index(drop=True)
print("Dataset after dropping missing values:")
print(df.isna().sum())
print(df.head(45))

Dataset after dropping missing values:
datetime                 0
open                     0
high                     0
low                      0
close                    0
volume                   0
log_return_5m            0
log_return_15m           0
rsi                      0
macd                     0
bollinger_hband          0
bollinger_lband          0
bollinger_mavg           0
target_log_return_5m     0
target_log_return_15m    0
dtype: int64
            datetime     open     high      low    close    volume  \
0   2017-09-01 07:35  4738.00  4740.00  4726.24  4740.00  0.942487   
1   2017-09-01 07:40  4745.00  4751.34  4728.13  4728.13  1.406108   
2   2017-09-01 07:45  4759.97  4766.99  4759.97  4766.99  0.481925   
3   2017-09-01 07:50  4766.99  4766.99  4723.00  4766.00  0.804373   
4   2017-09-01 07:55  4767.00  4767.00  4730.01  4730.04  1.035040   
5   2017-09-01 08:00  4730.10  4766.88  4730.10  4766.88  0.872060   
6   2017-09-01 08:05  4735.01  4735.01  4735.01  4735

In [9]:
#Phase 8: Saving the cleaned dataset to a new CSV file
df.to_csv('/home/dipankar/Desktop/crypto_price_prediction/data/processed/BTCUSDT_5m_2017-09-01_to_2025-09-23_cleaned_data.csv', index=False)

In [10]:
print((df == 0).sum())

datetime                    0
open                        0
high                        0
low                         0
close                       0
volume                    470
log_return_5m            3220
log_return_15m            879
rsi                         0
macd                        0
bollinger_hband             0
bollinger_lband             0
bollinger_mavg              0
target_log_return_5m     3220
target_log_return_15m     879
dtype: int64
